In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_csv("financial_transaction.csv")
data.head()

,Transaction_Text,Label
0,IKEA furniture shopping | Ref:37c9a05e | Amoun...,Shopping
1,MakeMyTrip hotel booking | Ref:7fb0c9d1 | Amou...,Travel
2,Uber ride payment | Ref:f3653a0a | Amount: INR...,Travel
3,Flipkart electronics purchase | Ref:ab027ce0 |...,Shopping
4,HDFC home loan EMI | Ref:44608adc | Amount: IN...,EMI


In [3]:
X = data.drop('Label' , axis=1)
y = data['Label']

# Model 1:-
It categories the raw bank statement csv file into lables:-

In [12]:
import re
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


@dataclass
class TxnPrediction:
    cleaned_text: str
    predicted_category: str
    confidence: float
    needs_review: bool
    top_k: List[Tuple[str, float]]


class ProductionTransactionClassifier:
    def __init__(self, confidence_threshold=0.72):
        self.confidence_threshold = confidence_threshold
        self.model = None

    @staticmethod
    def clean_text(raw_text: str) -> str:
        text = str(raw_text).upper().strip()
        text = re.sub(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+", " ", text)
        text = re.sub(r"\b(UPI|IMPS|NEFT|RTGS|POS|ATM|DR|CR|DEBIT|CREDIT|REF|RRN|UTR|TXN|ID|NO)\b", " ", text)
        text = re.sub(r"\b\d+(\.\d+)?\b", " ", text)
        text = re.sub(r"[^A-Z\s/&-]", " ", text)
        text = re.sub(r"[/&_-]+", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    @staticmethod
    def merchant_group_key(raw_text: str) -> str:
        text = ProductionTransactionClassifier.clean_text(raw_text)
        stop = {
            "OUT", "TO", "FROM", "ONLINE", "INDIA", "PAYMENT",
            "MONTHLY", "EXTRA", "DELIVERY", "TRIPS", "MEAL",
            "PREMIUM", "CREDITED"
        }
        tokens = [t for t in text.split() if t not in stop]
        return " ".join(tokens[:3]) if tokens else "UNKNOWN"

    @staticmethod
    def weak_label(raw_text: str) -> str:
        t = str(raw_text).upper()

        if re.search(r"\bSALARY\b", t):
            return "Salary/Income"
        if re.search(r"\bHOME LOAN\b|\bLOAN EMI\b|\bEMI\b", t):
            return "Loan EMI"
        if re.search(r"\bMUTUAL FUND\b|\bSIP\b|\bFIXED DEPOSIT\b|\bGROWW\b", t):
            return "Investments"
        if re.search(r"\bCREDIT CARD\b|\bCARD MONTHLY DUE\b", t):
            return "Credit Card Payment"
        if re.search(r"\bNETFLIX\b|\bSPOTIFY\b|\bICLOUD\b", t):
            return "Subscriptions"
        if re.search(r"\bSWIGGY INSTAMART\b|\bBLINKIT\b|\bKIRANA\b|\bGROCERY\b", t):
            return "Groceries"
        if re.search(r"\bZOMATO\b|\bDOMINOS\b|\bMCDONALDS\b|\bBURGER KING\b|\bSTARBUCKS\b|\bBARBEQUE\b|\bCAFE\b", t):
            return "Food & Dining"
        if re.search(r"\bUBER\b|\bOLA\b|\bPETROL\b|\bFUEL\b|\bMAKEMYTRIP\b", t):
            return "Travel & Fuel"
        if re.search(r"\bPVR\b|\bCINEMA\b|\bMOVIE\b", t):
            return "Entertainment"
        if re.search(r"\bAMAZON\b|\bMYNTRA\b|\bZARA\b|\bDECATHLON\b|\bFASHION\b|\bSHOPPING\b", t):
            return "Shopping"
        if re.search(r"\bJIO\b|\bRECHARGE\b|\bELECTRICITY\b|\bWATER\b|\bINSURANCE\b", t):
            return "Bills & Utilities"
        if re.search(r"\bTRANSFER TO\b|\bSENT TO\b|\bIMPS OUT TO\b|\bIFT OUT TO\b", t):
            return "Personal Transfer"

        return "Uncategorized"

    @staticmethod
    def add_features(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["cleaned_text"] = out["Description"].apply(ProductionTransactionClassifier.clean_text)
        out["amount"] = pd.to_numeric(out["Amount"], errors="coerce").fillna(0.0)
        out["log_amount"] = np.log1p(np.abs(out["amount"]))
        out["is_credit"] = out["Description"].str.upper().str.contains(r"\bCREDIT|CREDITED|CR\b", regex=True).astype(int)
        out["has_emi"] = out["Description"].str.upper().str.contains(r"\bEMI|LOAN\b", regex=True).astype(int)
        out["has_investment"] = out["Description"].str.upper().str.contains(r"\bSIP|MUTUAL FUND|FIXED DEPOSIT|GROWW\b", regex=True).astype(int)
        out["has_transfer"] = out["Description"].str.upper().str.contains(r"\bTRANSFER|SENT TO|IMPS OUT|IFT OUT\b", regex=True).astype(int)
        return out

    def build_pipeline(self):
        text_features = FeatureUnion([
            ("word_tfidf", Pipeline([
                ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True))
            ])),
            ("char_tfidf", Pipeline([
                ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True))
            ]))
        ])

        preprocessor = ColumnTransformer([
            ("text", text_features, "cleaned_text"),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler())
            ]), ["amount", "log_amount", "is_credit", "has_emi", "has_investment", "has_transfer"])
        ])

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            # REMOVED multi_class PARAMETER:
            ("classifier", LogisticRegression(solver="lbfgs", max_iter=500))
        ])

        return pipeline

In [13]:
import pandas as pd
import numpy as np

# ==========================================
# 1. MAKE SURE YOUR REFACTORED CLASS DEFINITION IS HERE
# ==========================================
# (Ensure your updated build_pipeline has solver="lbfgs" and no multi_class argument)

if __name__ == "__main__":
    # 1. Load your local transaction file
    # Ensure your file has "Description" and "Amount" columns
    input_filename = "testing.csv"
    output_filename = "predicted_transactions.csv"
    
    print(f"--- Loading data from {input_filename} ---")
    df = pd.read_csv(input_filename)
    
    # 2. Initialize your production classifier
    classifier = ProductionTransactionClassifier(confidence_threshold=0.72)
    
    # 3. Bootstrap training labels using the rule-based weak_label logic
    print("--- Simulating labels via keyword logic ---")
    df["Target"] = df["Description"].apply(classifier.weak_label)
    
    # 4. Feature engineering and preparation
    fe_data = classifier.add_features(df)
    
    # 5. Initialize and train the machine learning pipeline
    print("--- Training the Machine Learning Pipeline ---")
    classifier.model = classifier.build_pipeline()
    classifier.model.fit(fe_data, fe_data["Target"])
    
    # 6. Extract classification probability distribution across all categories
    print("--- Computing model predictions & confidence metrics ---")
    probabilities = classifier.model.predict_proba(fe_data)
    classes = classifier.model.classes_
    
    # Arrays to store tracking attributes row-by-row
    pred_categories = []
    confidences = []
    needs_review_flags = []
    
    for prob_dist in probabilities:
        # Match class names to confidence intervals and sort high -> low
        class_probs = sorted(zip(classes, prob_dist), key=lambda x: x[1], reverse=True)
        best_class, best_prob = class_probs[0]
        
        pred_categories.append(best_class)
        confidences.append(best_prob)
        # Tag rows where the algorithm is guessing or uncertain
        needs_review_flags.append(best_prob < classifier.confidence_threshold)
        
    # 7. Map metrics back onto original structure columns
    df["Cleaned_Text"] = fe_data["cleaned_text"]
    df["Predicted_Category"] = pred_categories
    df["Confidence_Score"] = confidences
    df["Needs_Review"] = needs_review_flags
    
    # Remove the internal 'Target' temporary calculation column before saving
    final_export_df = df.drop(columns=["Target"])
    
    # 8. Save output directly to file disk
    final_export_df.to_csv(output_filename, index=False)
    print(f"--- Process Finished successfully! Output saved to: '{output_filename}' ---")

--- Loading data from testing.csv ---
--- Simulating labels via keyword logic ---
--- Training the Machine Learning Pipeline ---
--- Computing model predictions & confidence metrics ---
--- Process Finished successfully! Output saved to: 'predicted_transactions.csv' ---


In [10]:
import pandas as pd
import numpy as np

# ==========================================
# 1. MAKE SURE YOUR REFACTORED CLASS DEFINITION IS HERE
# ==========================================
# (Ensure your updated build_pipeline has solver="lbfgs" and no multi_class argument)

if __name__ == "__main__":
    # 1. Load your local transaction file
    # Ensure your file has "Description" and "Amount" columns
    input_filename = "testing.csv"
    output_filename = "predicted_transactions.csv"
    
    print(f"--- Loading data from {input_filename} ---")
    df = pd.read_csv(input_filename)
    
    # 2. Initialize your production classifier
    classifier = ProductionTransactionClassifier(confidence_threshold=0.72)
    
    # 3. Bootstrap training labels using the rule-based weak_label logic
    print("--- Simulating labels via keyword logic ---")
    df["Target"] = df["Description"].apply(classifier.weak_label)
    
    # 4. Feature engineering and preparation
    fe_data = classifier.add_features(df)
    
    # 5. Initialize and train the machine learning pipeline
    print("--- Training the Machine Learning Pipeline ---")
    classifier.model = classifier.build_pipeline()
    classifier.model.fit(fe_data, fe_data["Target"])
    
    # 6. Extract classification probability distribution across all categories
    print("--- Computing model predictions & confidence metrics ---")
    probabilities = classifier.model.predict_proba(fe_data)
    classes = classifier.model.classes_
    
    # Arrays to store tracking attributes row-by-row
    pred_categories = []
    confidences = []
    needs_review_flags = []
    
    for prob_dist in probabilities:
        # Match class names to confidence intervals and sort high -> low
        class_probs = sorted(zip(classes, prob_dist), key=lambda x: x[1], reverse=True)
        best_class, best_prob = class_probs[0]
        
        pred_categories.append(best_class)
        confidences.append(best_prob)
        # Tag rows where the algorithm is guessing or uncertain
        needs_review_flags.append(best_prob < classifier.confidence_threshold)
        
    # 7. Map metrics back onto original structure columns
    df["Cleaned_Text"] = fe_data["cleaned_text"]
    df["Predicted_Category"] = pred_categories
    df["Confidence_Score"] = confidences
    df["Needs_Review"] = needs_review_flags
    
    # Remove the internal 'Target' temporary calculation column before saving
    final_export_df = df.drop(columns=["Target"])
    
    # 8. Save output directly to file disk
    final_export_df.to_csv(output_filename, index=False)
    print(f"--- Process Finished successfully! Output saved to: '{output_filename}' ---")

--- Loading data from testing.csv ---
--- Simulating labels via keyword logic ---
--- Training the Machine Learning Pipeline ---
--- Computing model predictions & confidence metrics ---
--- Process Finished successfully! Output saved to: 'predicted_transactions.csv' ---


# 